# 03 - WESAD preprocessing (RUN LOCALLY)
WESAD (~2.5 GB) is **not** in the project ZIP and was **not available** when this project was generated, so this notebook was **not executed** by the author. Set `WESAD_DIR` and run the cells.

WESAD is a stress/affect dataset. Its labels (baseline / stress / amusement) are never merged into the PHQ-9 depression targets.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 80)
from src.utils import load_config, reports_dir, outputs_dir
cfg = load_config()          # uses $DEPRESSION_CONFIG if set, else configs/config.yaml
print("config:", cfg["_config_path"])

In [ ]:
WESAD_DIR = Path(cfg['paths']['wesad_dir'])    # edit configs/config.yaml or overwrite here, e.g. Path('/Users/me/data/WESAD')
print(WESAD_DIR, 'exists:', WESAD_DIR.is_dir())

## Step 1 - structure audit only (fast)

In [ ]:
from src.wesad_preprocessing import run
OUT = outputs_dir(cfg, 'wesad')
rep = run(WESAD_DIR, OUT, cfg, audit_only=True)
print('subjects found:', rep['subjects_found'])
print('missing expected:', rep['missing_expected_subjects'])
pd.read_csv(OUT / 'wesad_signal_inventory.csv').groupby(['device', 'signal']).agg(fs_hz=('inferred_fs_hz', 'median'), nonfinite=('n_nonfinite', 'sum'), flatline_subjects=('flatline', 'sum'))

## Step 2 - window feature extraction (one subject at a time)
Tip: try `subjects=['S2']` first.

In [ ]:
rep = run(WESAD_DIR, OUT, cfg, subjects=None, audit_only=False)
print('windows:', rep['total_windows'], '| features:', rep['n_features'])
rep['label_distribution_total']

In [ ]:
feats = pd.read_parquet(OUT / 'wesad_windows_features.parquet')
feats.groupby('label_name').size()

## Step 3 - stress experiment preparation (supplementary, NOT depression)
Use **leave-one-subject-out** (`loso_folds.csv`). Do not normalise per subject with label-derived baselines.

In [ ]:
import json
cols = json.load(open(OUT / 'feature_columns.json'))['feature_columns']
print(len(cols), 'feature columns; subject is kept only for splitting')
feats[['subject', 'label_binary_stress'] + cols[:5]].head()